In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import scipy as sp
import xarray as xr
import seaborn as sns
import os
import traceback
from ptsa.data.filters import ResampleFilter, ButterworthFilter, MorletWaveletFilter
from ptsa.data.timeseries import TimeSeries
save_path = '/data7/rsa_class'
import sys
sys.path.append("dependencies/bidsreader")
from bidsreader import CMLBIDSReader as BIDSReader
from bidsreader import mne_epochs_to_ptsa

# Computing Correlations Using xarray

First we are going to try to demonstrate how to compute a correlation by "hand" and then how to compute it efficiently using xarray.

As a simple example, imagine we are looking at the brain while people look at states like in the Chipman and Shepard (1970) paper from lecture). Generate a dataset where we observe 10 neural features across 15 states

In [2]:
A = np.diag(np.ones(15))  # a diagonal covariance matrix

In [3]:
A

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0.

Add in a couple true correlations across "states"

In [4]:
A[1, 2] = A[2, 1] = .5
A[6, 8] = A[6, 8] = .8
A

array([[1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 1. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0.5, 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0.8, 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. ,
        0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. ,
       

In [5]:
features = np.random.multivariate_normal(size=10, mean=np.zeros(15), cov=A).T

/tmp/ipykernel_51802/3651426059.py:1: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  features = np.random.multivariate_normal(size=10, mean=np.zeros(15), cov=A).T


Now we want to look at two sets of features and see how similar they are

In [6]:
features[1], features[2]

(array([-0.19713603, -1.23268788,  4.24744025, -0.98691831,  0.46402489,
        -0.86205585,  1.72794717,  1.22953926, -2.46464822,  0.62291206]),
 array([-0.49407518, -2.04284526,  1.83017635, -0.49400003,  1.67508499,
         1.41503624,  1.14436882, -0.44984227, -0.42753734,  1.58506255]))

Generally feature 1 is negative when feature 2 is negative and positive when feature 2 is positive so they seem pretty similar (this is because we made them similar when we generated the covariance matrix). This should be reflected when we compute their correlation. [Pearson's correlation](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient) is the covariance of the two variables divided by the standard deviation of each multiplied together.

Covariance is the expectation (aka expected value aka mean) of (feature 1 - mean(feature 1)) * (feature 2 - mean(feature 2). This measure is going to be high if feature 1 is usually above its mean when feature 2 is also above its mean. Lets compute it

In [7]:
np.mean(features[1]), np.mean(features[2])

(0.2548417332596765, 0.37414288473802293)

In [8]:
(features[1] - np.mean(features[1])) * (features[2] - np.mean(features[2]))

array([ 0.39241526,  3.59534144,  5.81335705,  1.07802519,  0.27213517,
       -1.16257127,  1.13462401, -0.8031363 ,  2.18016132,  0.44570359])

Now we take the average of that value (but its a sample statistic so we use n-1 in the denominator, instead of n=10)

In [9]:
feat_cov = (np.sum((features[1] - np.mean(features[1])) * 
        (features[2] - np.mean(features[2]))) / (10 - 1)) 
feat_cov 

1.4384506063990434

We can check it against `numpy`'s version

In [10]:
np.cov(features[1], features[2])

array([[3.53802591, 1.43845061],
       [1.43845061, 1.73627404]])

Now just divide that by the sample standard deviations

In [11]:
feat_cov / (np.std(features[1], ddof=1) * np.std(features[2], ddof=1))

0.5803704785645805

And check that against numpy's version of correlation

In [12]:
np.corrcoef(features[1], features[2])

array([[1.        , 0.58037048],
       [0.58037048, 1.        ]])

Now we want a way to compute these correlations for all of the states and keep track of all of the state info. Like we do with EEG, we'll store it using an xarray DataArray

In [13]:
states_da = xr.DataArray(data=features, 
             dims=["states", "features"], 
             coords={'features': ['feat_' + str(i) for i in range(10)],
                     'states': ['Minn.', 'Ore.', 'W.V.', 'Colo.', 'Ala.', 'Ill.',
                               'Nev.', 'Nebr.', 'Okla.', 'Ida.', 'Fla.', 'La.', 
                                'S.C.', 'Mo.', 'Me.']})

In [14]:
states_da

<xarray.DataArray (states: 15, features: 10)>
array([[ 0.01862978,  0.88433344, -0.03135099, -1.28733158,  0.0840661 ,
         2.1115697 ,  0.97919973,  1.01957513,  0.59051167,  0.3992519 ],
       [-0.19713603, -1.23268788,  4.24744025, -0.98691831,  0.46402489,
        -0.86205585,  1.72794717,  1.22953926, -2.46464822,  0.62291206],
       [-0.49407518, -2.04284526,  1.83017635, -0.49400003,  1.67508499,
         1.41503624,  1.14436882, -0.44984227, -0.42753734,  1.58506255],
       [-0.82490605,  1.46020977,  0.45652902, -0.94889284,  0.9435855 ,
        -0.71300736, -0.37059606,  0.97911679, -1.51192633,  1.43991669],
       [ 0.30389076, -0.6624036 ,  0.07628553, -2.64348202, -0.20228316,
        -1.08461918,  1.6077903 , -0.71679087,  0.09473507, -1.55103357],
       [-0.3510753 ,  0.17979324, -0.61107808,  0.21091638, -0.59497094,
         0.46485279,  0.8078673 , -0.98505925,  0.03839108,  0.57525821],
       [-0.94931272,  1.39852854, -0.3016627 , -0.88670222,  0.58280123,
         0.88184726,  0.74478183,  0.87526327, -0.47999065,  0.78984581],
       [-1.34306976, -1.57070631,  0.48789808,  0.42117817,  2.28812004,
         0.49972629, -0.12590839, -0.10839224, -0.25380822,  1.15584972],
       [ 0.96226065,  1.98110447, -0.71354224, -0.91185307,  0.44260503,
         0.33094599,  3.40774713, -0.41042694, -2.83871594,  0.98084117],
       [-0.58539793, -0.60776974, -0.67771782,  0.439555  , -0.75466272,
        -0.28428417, -1.38645566, -0.24485169, -1.78695249, -0.62960238],
       [-1.54813443,  0.58220049, -0.25041191,  0.75571331, -1.12320246,
         0.18285482, -1.65964399, -0.52272526,  0.98861682, -1.74713037],
       [ 0.48176688, -1.44594374,  0.5459504 , -0.64310871, -1.26499205,
         1.15584098,  1.63275405,  1.3090859 , -0.7825251 , -2.6114677 ],
       [-0.14196228,  1.9726421 , -1.12236992,  1.36454354, -0.93481258,
        -0.48016169, -0.26298816, -1.34954505,  1.5895742 ,  0.18134634],
       [-0.09874419, -0.14724681,  0.2024156 ,  2.03149537,  0.3591185 ,
         1.3036681 ,  0.18740968,  1.30325058, -1.46379777, -0.59176702],
       [-0.09518322, -1.07137105, -0.35554333,  0.68808498,  0.70628101,
        -1.73556281, -1.3525989 ,  1.50113167, -1.33277041, -1.10123042]])
Coordinates:
  * features  (features) <U6 'feat_0' 'feat_1' 'feat_2' ... 'feat_8' 'feat_9'
  * states    (states) <U5 'Minn.' 'Ore.' 'W.V.' 'Colo.' ... 'S.C.' 'Mo.' 'Me.'

Now we can use the xr.corr function to compute the correlation of features between states but its not so straightforward because the states have the same names in both -- just plugging it in like this leads to the wrong answer because it matches the array along its dimensions. This leads to just correlates each state with itself, which is going to  always be 1

In [15]:
xr.corr(states_da, states_da, dim='features')

<xarray.DataArray (states: 15)>
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Coordinates:
  * states   (states) <U5 'Minn.' 'Ore.' 'W.V.' 'Colo.' ... 'S.C.' 'Mo.' 'Me.'

What we want is each state correlated with every other one. We need to rename the states variable to get the right answer

In [16]:
states_da2 = states_da.rename({'states': 'states2'})

In [17]:
states_corr = xr.corr(states_da, states_da2, dim='features')

In [18]:
states_corr

<xarray.DataArray (states: 15, states2: 15)>
array([[ 1.        , -0.08052038,  0.10885862,  0.10764088,  0.33225738,
         0.21223448,  0.69493292, -0.14838926,  0.27721111, -0.32005534,
        -0.04638815,  0.36463533, -0.22039511, -0.12645066, -0.51358131],
       [-0.08052038,  1.        ,  0.58037048,  0.39266989,  0.33010791,
        -0.31412295,  0.06545477,  0.26851785,  0.25335109,  0.04107224,
        -0.49890064,  0.34734903, -0.73439426,  0.1439573 ,  0.22507609],
       [ 0.10885862,  0.58037048,  1.        ,  0.07923804,  0.1627334 ,
         0.08732098,  0.0648277 ,  0.77954017,  0.07285057, -0.11973826,
        -0.49284601,  0.0985866 , -0.6666257 ,  0.02689803, -0.14551091],
       [ 0.10764088,  0.39266989,  0.07923804,  1.        , -0.11769662,
        -0.2242307 ,  0.68750858,  0.20812305,  0.39186459,  0.17105616,
        -0.33215157, -0.39599805, -0.24109953, -0.02733939,  0.22342716],
       [ 0.33225738,  0.33010791,  0.1627334 , -0.11769662,  1.        ,
        -0.03683222,  0.06896465, -0.23096411,  0.36178368, -0.77328453,
        -0.40624301,  0.46509954, -0.28821508, -0.48829672, -0.27288744],
       [ 0.21223448, -0.31412295,  0.08732098, -0.2242307 , -0.03683222,
         1.        ,  0.20154867, -0.07592869,  0.42739964, -0.17509445,
        -0.03521051, -0.15682956,  0.48457553, -0.13833015, -0.76613964],
       [ 0.69493292,  0.06545477,  0.0648277 ,  0.68750858,  0.06896465,
         0.20154867,  1.        ,  0.09342629,  0.53472905, -0.07091062,
...
         0.42739964,  0.53472905, -0.17102379,  1.        , -0.01877742,
        -0.61240773,  0.13193008, -0.12052183,  0.05684968, -0.24144493],
       [-0.32005534,  0.04107224, -0.11973826,  0.17105616, -0.77328453,
        -0.17509445, -0.07091062,  0.09717033, -0.01877742,  1.        ,
         0.13651723, -0.00382643, -0.09914123,  0.84024653,  0.51886763],
       [-0.04638815, -0.49890064, -0.49284601, -0.33215157, -0.40624301,
        -0.03521051, -0.15206886, -0.23548532, -0.61240773,  0.13651723,
         1.        , -0.06763249,  0.5761054 ,  0.13809673, -0.06974546],
       [ 0.36463533,  0.34734903,  0.0985866 , -0.39599805,  0.46509954,
        -0.15682956, -0.06521372, -0.25692286,  0.13193008, -0.00382643,
        -0.06763249,  1.        , -0.51074998,  0.40933964,  0.07107478],
       [-0.22039511, -0.73439426, -0.6666257 , -0.24109953, -0.28821508,
         0.48457553, -0.13034135, -0.45156604, -0.12052183, -0.09914123,
         0.5761054 , -0.51074998,  1.        , -0.29533652, -0.36759279],
       [-0.12645066,  0.1439573 ,  0.02689803, -0.02733939, -0.48829672,
        -0.13833015,  0.01048669,  0.1813314 ,  0.05684968,  0.84024653,
         0.13809673,  0.40933964, -0.29533652,  1.        ,  0.5101086 ],
       [-0.51358131,  0.22507609, -0.14551091,  0.22342716, -0.27288744,
        -0.76613964, -0.22600594,  0.2137554 , -0.24144493,  0.51886763,
        -0.06974546,  0.07107478, -0.36759279,  0.5101086 ,  1.        ]])
Coordinates:
  * states   (states) <U5 'Minn.' 'Ore.' 'W.V.' 'Colo.' ... 'S.C.' 'Mo.' 'Me.'
  * states2  (states2) <U5 'Minn.' 'Ore.' 'W.V.' 'Colo.' ... 'S.C.' 'Mo.' 'Me.'

In [19]:
states_corr_df = states_corr.to_dataframe('corr').reset_index()
states_corr_df

,states,states2,corr
0,Minn.,Minn.,1.000000
1,Minn.,Ore.,-0.080520
2,Minn.,W.V.,0.108859
3,Minn.,Colo.,0.107641
4,Minn.,Ala.,0.332257
...,...,...,...
220,Me.,Fla.,-0.069745
221,Me.,La.,0.071075
222,Me.,S.C.,-0.367593
223,Me.,Mo.,0.510109


This is the same as we calculated before!

In [20]:
states_corr_df.query('states == "Ore." and states2 == "W.V."')

,states,states2,corr
17,Ore.,W.V.,0.58037


# Computing correlations in EEG data

## Preprocess data

We first load the data and preprocess into power at different frequency bands both during encoding (300ms to 1300 ms after the word appears on screen) and right before retrieval (900 to 100ms before vocalization). In order to make signals comparable across encoding and retrieval, we standardize both of them by data from during the 10s countdown periods that preceded each list. This means that all of the data is relative to the countdown period (e.g. there is .5 standard deviations more power in the 3 Hz band than the average of the countdown period).

In [21]:
sub = 'R1045E'
task = 'catFR1'
ses = 0

enc_range_left_ms = 0.3
enc_range_right_ms = 1.3
morlet_reps = 6
freqs = np.logspace(np.log10(3), np.log10(180), 8)
resameplerate = 500
enc_buffer_time = (morlet_reps / 2) * (1 / min(freqs))

reader = BIDSReader(subject=sub, session=ses, task=task, root = '/data/LTP_BIDS/catFR1')

evs = reader.load_events(event_type="ieeg")

enc_evs = evs.query('trial_type == "WORD" and sample != -1')
enc_evs = enc_evs[enc_evs['list'] > 0]
enc_evs['category'] = enc_evs ['category'].str.lower()
enc_evs = enc_evs.iloc[:25]

eeg = reader.load_epochs(events=enc_evs, tmin=enc_range_left_ms-enc_buffer_time, tmax=enc_range_right_ms+enc_buffer_time, acquisition="monopolar")

eeg_ptsa = mne_epochs_to_ptsa(events_df=enc_evs, epochs=eeg)

Extracting EDF parameters from /data/LTP_BIDS/catFR1/sub-R1045E/ses-0/ieeg/sub-R1045E_ses-0_task-catFR1_acq-monopolar_ieeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading events from /data/LTP_BIDS/catFR1/sub-R1045E/ses-0/ieeg/sub-R1045E_ses-0_task-catFR1_events.tsv.
Reading channel info from /data/LTP_BIDS/catFR1/sub-R1045E/ses-0/ieeg/sub-R1045E_ses-0_task-catFR1_acq-monopolar_channels.tsv.
Reading electrode coords from /data/LTP_BIDS/catFR1/sub-R1045E/ses-0/ieeg/sub-R1045E_ses-0_task-catFR1_space-MNI152NLin6ASym_electrodes.tsv.
Used Annotations descriptions: ['COUNTDOWN_END', 'COUNTDOWN_START', 'DISTRACT_END', 'DISTRACT_START', 'ORIENT', 'PRACTICE_DISTRACT_END', 'PRACTICE_DISTRACT_START', 'PRACTICE_REC_END', 'PRACTICE_REC_START', 'PRACTICE_WORD', 'PROB', 'REC_END', 'REC_START', 'REC_WORD', 'SESS_END', 'SESS_START', 'START', 'STOP', 'TRIAL', 'WORD']
Not setting metadata
25 matching events found
No baseline correction applied
0 projecti

ExternalLibraryError: TypeError: mne_epochs_to_ptsa() got an unexpected keyword argument 'events_df'

In [ ]:
# centering signal within event
# reduce edge effects / ringing in later processing steps:
eeg_ptsa = eeg_ptsa.astype(float) - eeg_ptsa.mean('time')
# filter out line noise at 60 Hz
eeg_ptsa = ButterworthFilter(filt_type='stop', 
                        freq_range=[58, 62], 
                        order=4).filter(timeseries=eeg_ptsa)
# compute spectral features!
wf = MorletWaveletFilter(freqs=freqs, width=morlet_reps, output='power', complete=True)
# freqs, events, elecs, and time
powers = wf.filter(eeg_ptsa)

In [ ]:
powers.data = np.log10(powers.data)
powers

In [ ]:
print('resample rate', resameplerate)
powers = ResampleFilter(resamplerate=resameplerate).filter(
    timeseries=powers)
powers = powers.remove_buffer(enc_buffer_time)

# reshape as events x features
powers = powers.stack(features=("channel", "frequency"))

In [ ]:
powers.indexes['event'].names

In [ ]:
# need to remove stim_params because of issues with converting to pandas DataFrame
if 'stim_params' in powers.indexes['event'].names:
    powers = powers.assign_coords(
        {"event": powers.indexes['event'].droplevel('stim_params')}
    )

## Preprocess recall events

In [ ]:
rec_range_left_ms = -0.9
rec_range_right_ms = -0.1
rec_buffer_time = 0.76

In [ ]:
eeg_ptsa

In [ ]:
rec_evs = evs.query('trial_type == "REC_WORD" and sample != -1')
rec_evs = rec_evs[rec_evs['list'] > 0]
rec_evs['category'] = rec_evs ['category'].str.lower()
rec_evs = rec_evs.iloc[:25]
# get inter-retreival times since previous recall
rec_evs['pirt'] = rec_evs.groupby(['session', 'list'])['response_time'].diff().fillna(
    rec_evs['response_time'])
rec_evs['repeat'] = rec_evs.duplicated(subset=['session', 'list', 'item_name'])
rec_evs['outpos'] = rec_evs.groupby(['subject', 'session', 'list']).cumcount()
# only include recalls at least 1500 ms away and no repeats
rec_evs = rec_evs.query('pirt > 1.5 and repeat == 0')

**Load EEG using PTSA just like above**

Because recalls might run into each other, in order to create a buffer, we need to use what's known as a mirror buffer. This ensures that future vocalizations don't screw up the buffer period

In [ ]:
eeg_ptsa['time'] = eeg_ptsa['time'] # PTSA time scale is in seconds instead of ms
eeg_ptsa = eeg_ptsa.add_mirror_buffer(rec_buffer_time)
eeg_ptsa['time'] = eeg_ptsa['time'] 

**Preprocess data just like above**

## Preprocess countdown events

In [ ]:
countdown_rel_start = 0
countdown_rel_stop = 10
countdown_resameplerate = 1

sub = 'R1045E'
sess = 0
countdown_events = pd.read_csv(save_path+'/'+exp+'_countdown_evs.csv')
countdown_evs = countdown_events.query('subject == @sub and session == @sess')

**Preprocess data for countdown events just like above**

## Compute RSA

Now we have to compute the RSA. This basically involves loading the outputs of the `compute_features` function above and using the xarray `corr` function which will compute the correlation matrix between two data arrays and hold on to all the relevant information for us. There are two tricky things going on here. One is that we need to normalize the encoding and retrieval time features by the countdown features. we have a function called `normalize_features` to do that. We also need to change the names of the event information so that they don't match. If they do, the `corr` function will assume they are referring to the same informaiton and will not compute a correlation between them. In order to distinguish the item_name at retrieval and at encoding, we append the event `type` onto the name of each column. If we are computing the correlation between two events of the same `type` (e.g. each encoding event to other encoding events), we add a 2 on the end to distinguish them.

In [ ]:
experiment = "catFR1"
encoding_type = "WORD"
comparison_type = "WORD"
countdown_normalize = 1

In [ ]:
def normalize_features(pows, save_path, countdown_normalize=True):
    subject, session = pows.event.subject.values[0], pows.event.session.values[0]
    if countdown_normalize:
        countdown_fp = save_path + '/' + subject + '_' + str(session) + '_COUNTDOWN_START_feats.h5'
        countdown_pows = TimeSeries.from_hdf(countdown_fp)
        countdown_pows['samplerate'] = pows['samplerate']
        pows = (pows - countdown_pows.mean(['event', 'time'])) / countdown_pows.std(['event', 'time']) 
    else:
        pows = pows.reduce(func=sp.stats.zscore, dim='event', keep_attrs=True, ddof=1)
    return pows

def set_event_names(pows, period_type, col='event', copy=False):
    if copy:
        pows = pows.copy()
    events_mi = pows[col].to_index()
    events_mi = events_mi.rename(
        [name+'_'+period_type for name in events_mi.names])
    pows[col] = events_mi
    pows = pows.rename({col: col+'_'+period_type})
    return pows

In [ ]:
print("Processing subject:", sub, "session:", sess)

In [ ]:
encoding_fp = save_path + '/' + sub + '_' + str(sess) + '_' + encoding_type + '_feats.h5'
encoding_pows = TimeSeries.from_hdf(encoding_fp)
encoding_pows = normalize_features(encoding_pows, save_path, countdown_normalize=True)

In [ ]:
#need to rename event index to keep track of things for correlation matrix
comparison_pows = set_event_names(encoding_pows, encoding_type+'2', col='event', copy=True)
encoding_pows = set_event_names(encoding_pows, encoding_type, col='event')

In [ ]:
corr_arr = xr.corr(encoding_pows, comparison_pows, dim='features')

We now convert it to a dataframe for easy use later. We transform the correlation into `corr_z`, the [Fisher](https://academic.oup.com/biomet/article/10/4/507/203628) [$z$-transformed](https://en.wikipedia.org/wiki/Fisher_transformation) correlation between two encoding periods. We transform it because if we average together a bunch of correlations our result will be biased relative to the true average correlation. The transformation to some extent corrects for this bias (asymptotically). 

In [ ]:
corr_df = corr_arr.to_dataframe('corr').reset_index(drop=True)
corr_df['corr_z'] = np.arctanh(corr_df['corr'])

In [ ]:
corr_df

For the encoding-to-encoding RSA, we can cut things down a bit by enforcing that the `_WORD` events come before `_WORD2`. This is fine since the full correlation matrix is symmetric so we're just taking the lower triangle.

In [ ]:
corr_df = corr_df.query('((list_WORD2 > list_WORD) or ' +
                              '((list_WORD2 == list_WORD) and ' +
                              '(serialpos_WORD2 > serialpos_WORD)))'
                             )

Now we do the same thing comparing recall to encoding

In [ ]:
comparison_type = "REC_WORD"

In [ ]:
comparison_fp = save_path + '/' + sub + '_' + str(sess) + '_' + comparison_type + '_feats.h5'
comparison_pows = TimeSeries.from_hdf(comparison_fp)
comparison_pows = normalize_features(comparison_pows, save_path, countdown_normalize=countdown_normalize)
comparison_pows = set_event_names(comparison_pows, comparison_type, col='event')

**Compute RSA comparing encoding to recall events just like above**

# Analyzing RSA

Having computed the correlation matrices, now we need to analyze it to see how it varies with, for instance, whether two items are from the same category. Fortunately for you, I've already computed the correlation matrices for several subjects so you don't need to wait forever or use tons of compute resources to do it.

# Across-list category similarity

We'll start by looking at a classic result from [Haxby et al. (2001)](https://www.science.org/doi/10.1126/science.1063736). They presented subjects with images from eight different categories:

<center>
    <img src="figures/HaxbyEtal01_fig2.png" width=400>
</center>

They then compared the neural representations, finding that the patterns in "object-selective cortex" were more similar when looking at two images from the same category than across categories!

<center>
    <img src="figures/HaxbyEtal01_fig4.png" width=400>
</center>

[Kriegeskorte et. al. (2008)](https://www.sciencedirect.com/science/article/pii/S0896627308009434) conducted a similar study using fMRI in humans and neural recordings in monkeys, showing that the categorical representation of images in Inferior Temporal (IT) cortex was similar in both:

<center>
    <img src="figures/KrieEtal08_fig1.png" width=800>
</center>

Side note: you can actually analyze the original data from this study as it's freely available along with the [pyMVPA package](http://www.pymvpa.org/) (see [here](http://www.pymvpa.org/tutorial_classifiers.html) for an example showing you how to analyze this data). pyMVPA was one of the original python packages, developed by people from Jim Haxby's and Ken Norman's labs, for performing modern cognitive neuroscience analyses. The Haxby data is also available as part of [nilearn](https://nilearn.github.io/stable/decoding/decoding_intro.html?highlight=haxby), another python package for processing fMRI data.

pyMVPA was to some extent superseded by [Brainiak](brainiak.org), developed by many labs at Princeton (including Ken Norman's). Brainiak has its own RSA tutorial [here](https://brainiak.org/tutorials/06-rsa/), using data from the Kriegeskorte study. In addition, Brainiak and nilearn both have a number of other tutorials which are worth looking at if you're interested in modern fMRI methods (some of which have yet to be applied to EEG!).

That being said, if you look at these tutorials, you'll see that there's nothing fancy you need to do basic RSA with a correlation metric, just the simple correlation matrix computations from numpy or xarray, so no need to install those packages aside from them being an easy way to access these older datasets! 

Because this class is focused on EEG, rather than getting into the details of fMRI, we'll try to look for similar effects in one of our intracranial EEG datasets. We have a collected data from a number of patients doing a categorized free recall task (first reported [here](https://psycnet.apa.org/record/2018-66881-001), although we have a lot more data collected now!). This task is the same as a standard verbal free recall task except that the words on each list come a smaller number of categories. In this particular version, it also has a special structure, where each list has three categories and the items were presented in pairs of same-category words. The structure looks like this:

<center>
    <img src="figures/WeidEtal19_fig1_catFR.png" width=400>
</center>

Within a session, subjects collected early on would see at most 300 words, 12 words from each of 25 categories. After collecting data from about 120 subjects, we decided to cut the sessions in half but added another category so subjects saw 312 words from 26 categories across two "half sessions." Because the words on the lists come from a small set of categories, we can try to do a similar analysis on these data as they did in the fMRI papers mentioned above, investigating whether the representations of pairs of same category words are more similar than pairs from different categories. In most RSA analyses (like above) the focus is on regions of the brain that are known to respond to the type of stimuli in question. In intracranial EEG though, we don't get to choose where in the brain we are recording from and it's not consistent across patients. Therefore, in the analyses here, to simplify things for this class, we will just use all of the electrodes we get for each patient and look at their similarity. Perhaps surprisingly, as we'll see, this seems to work ok! 

In a related but slightly more complex analysis, [Kragel et al. (2021)](https://www.nature.com/articles/s41467-021-24393-1) showed that it was possible to decode the category of a word in this task from just electrodes within two specific brain regions (Posterior Medial, PM, and Anterior Temporal, AT):

<center>
    <img src="figures/KragEtal21_fig2c.png" width=400>
</center>

In addition, using data from an uncategorized free recall task, [Manning et al. (2012)](https://www.jneurosci.org/content/32/26/8871.short) showed that similarity of neural activity seemed to be related to a measure of semantic similarity for some subjects:

<center>
    <img src="figures/MannEtal12_fig3.png" width=400>
</center>

## Analysis

One tricky thing about doing this analysis (as we discussed in lecture) is that the EEG signal is really influenced by time, i.e. things that happen nearby also have similar patterns of activity. In the Haxby study, they handled this by only comparing representations across runs. Similarly, in this first analysis, we will only look at across list comparisons.

In [ ]:
raw_WORD_rsa_df = pd.read_csv(save_path + '/raw_WORD_rsa_df.csv')
raw_REC_WORD_rsa_df = pd.read_csv(save_path + '/raw_REC_WORD_rsa_df.csv')

In [ ]:
across_list_WORD_rsa_df = raw_WORD_rsa_df.query('(list_WORD2 > list_WORD)') 
within_list_WORD_rsa_df = raw_WORD_rsa_df.query('((list_WORD2 == list_WORD) and (serialpos_WORD2 > serialpos_WORD))')

Check out what information is contained in this dataframe. `corr_z` is going to be the relevant column for investigating similarity. 
Beyond that, the information contained in the `_WORD` columns refers to the word in the pair that occurred earlier in the session (on a prior list), `_WORD2` is the later word.

In [ ]:
across_list_WORD_rsa_df

In order to start looking at this we need to start averaging. We want to be careful about how we do this because the data isn't independent. There's a lot of structure since each word occurred on a list on a session for a subject. We'll first look only at comparisons between the same category on a specific pair of lists in a session and get the average representation there. 

In [ ]:
across_list_WORD_rsa_list_df = across_list_WORD_rsa_df.groupby(
    ['subject_WORD', 'session_WORD', 'list_WORD', 'list_WORD2', 'category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z': 'mean'})

In [ ]:
across_list_WORD_rsa_list_df

If you haven't seen this code pattern yet, it's super useful and comes up all the time in data science. It's called a [split-apply-combine pattern](https://www.jstatsoft.org/article/view/v040i01) because we split up the data into chunks, run some code on each of them (here taking the mean correlation), and then recombine them again. You can see more about how this works in `pandas` [here](https://pandas.pydata.org/docs/user_guide/groupby.html). We then continue taking the average within session, and subject until we finally get to the average similarity between items from a pair of categories in our dataset.

In [ ]:
across_list_WORD_rsa_session_df = across_list_WORD_rsa_list_df.groupby(
    ['subject_WORD', 'session_WORD', 'category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z': 'mean'})
across_list_WORD_rsa_subject_df = across_list_WORD_rsa_session_df.groupby(
    ['subject_WORD', 'category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z': 'mean'})
across_list_WORD_rsa_category_df = across_list_WORD_rsa_session_df.groupby(
    ['category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z': 'mean'})

As we mentioned before, some subjects saw 26 categories while some only saw 25. To make this analysis more straightforward, we'll remove the 26th category, "fabrics" from the analysis.

In [ ]:
across_list_WORD_rsa_category_df = across_list_WORD_rsa_category_df.query('(category_WORD != "fabric") and ' + 
                                                                            '(category_WORD2 != "fabric")')

Finally we'll reshape this data into a matrix so we can plot it like the Haxby/Kriegeskorte analyses above. Note that this matrix isn't symmetric -- the y-axis corresponds to the earlier word and the x-axis to the later word.

In [ ]:
across_list_WORD_rsa_category_mat = across_list_WORD_rsa_category_df.pivot_table(
    index=['category_WORD'], columns=['category_WORD2'], 
    values='corr_z')

In [ ]:
g = sns.heatmap(across_list_WORD_rsa_category_mat)
g.set(xlabel='Later Item', ylabel='Earlier Item')

This doesn't look like much -- the diagonal structure isn't super apparent. One reason is that there is a lot of uncontrolled variance here. Some subjects' neural activity is more category driven than others so the average similarities vary a lot by subject. It also varies by session and list. Use the split apply combine approach to see what the variability in average similarity looks like across subjects. Use seaborn [displot](https://seaborn.pydata.org/generated/seaborn.displot.html) to investigate

What we are really interested in is, within a subject and a pair of lists, do same category words look more similar than across category? Therefore we'll subtract the average similarity within a pair of lists from the `corr_z` column. This gives us only the within-list-pair variance. This is often a [good strategy](https://www.tqmp.org/RegularArticles/vol01-1/p042/p042.pdf) for visualizing effects in studies like this where there is hierarchical structure in the design (subjects/sessions/lists). We'll store this in the `corr_z_list_adj` column

In [ ]:
across_list_WORD_rsa_df['corr_z_list_mean'] = across_list_WORD_rsa_df.groupby(
    ['subject_WORD', 'session_WORD', 'list_WORD', 'list_WORD2'])['corr_z'].transform('mean')
across_list_WORD_rsa_df['corr_z_list_adj'] = (across_list_WORD_rsa_df['corr_z'] - 
                                                     across_list_WORD_rsa_df['corr_z_list_mean'])
within_list_WORD_rsa_df['corr_z_list_mean'] = within_list_WORD_rsa_df.groupby(
    ['subject_WORD', 'session_WORD', 'list_WORD'])['corr_z'].transform('mean')
within_list_WORD_rsa_df['corr_z_list_adj'] = (within_list_WORD_rsa_df['corr_z'] - 
                                                     within_list_WORD_rsa_df['corr_z_list_mean'])

In [ ]:
across_list_WORD_rsa_list_df = across_list_WORD_rsa_df.groupby(
    ['subject_WORD', 'session_WORD', 'list_WORD', 'list_WORD2', 'category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z_list_adj': 'mean'})
across_list_WORD_rsa_session_df = across_list_WORD_rsa_list_df.groupby(
    ['subject_WORD', 'session_WORD', 'category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z_list_adj': 'mean'})
across_list_WORD_rsa_subject_df = across_list_WORD_rsa_session_df.groupby(
    ['subject_WORD', 'category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z_list_adj': 'mean'})
across_list_WORD_rsa_category_df = across_list_WORD_rsa_session_df.groupby(
    ['category_WORD', 'category_WORD2'], as_index=False
).agg({'corr_z_list_adj': 'mean'})

In [ ]:
across_list_WORD_rsa_category_df = across_list_WORD_rsa_category_df.query(
    '(category_WORD != "fabric") and ' + 
    '(category_WORD2 != "fabric")')
across_list_WORD_rsa_category_mat = across_list_WORD_rsa_category_df.pivot_table(
    index=['category_WORD'], columns=['category_WORD2'], 
    values='corr_z_list_adj')

In [ ]:
g = sns.heatmap(across_list_WORD_rsa_category_mat)
g.set(xlabel='Later Item', ylabel='Earlier Item')

This still doesn't look amazing, you can't really see a diagonal unless you really squint. It's challenging to see effects like this at some level because of the range of electrodes you get across patients. However, we can focus our analysis a little more. Let's see if same category comparisons are more similar than across in general.

In [ ]:
across_list_WORD_rsa_df['same_category'] = across_list_WORD_rsa_df['category_WORD'] == across_list_WORD_rsa_df['category_WORD2']
across_list_WORD_rsa_list_df = across_list_WORD_rsa_df.groupby(
    ['subject_WORD', 'session_WORD', 'list_WORD', 'list_WORD2', 'same_category'], as_index=False
).agg({'corr_z_list_adj': 'mean'})
across_list_WORD_rsa_session_df = across_list_WORD_rsa_list_df.groupby(
    ['subject_WORD', 'session_WORD', 'same_category'], as_index=False
).agg({'corr_z_list_adj': 'mean'})
across_list_WORD_rsa_subject_df = across_list_WORD_rsa_session_df.groupby(
    ['subject_WORD', 'same_category'], as_index=False
).agg({'corr_z_list_adj': 'mean'})

In [ ]:
g = sns.catplot(x='same_category', y='corr_z_list_adj', 
            data=across_list_WORD_rsa_subject_df, kind='point')
g.set(xlabel='Same Category', ylabel='List-centered $z$-transformed Correlation')

Now we can see that there is in fact a difference, although it is somewhat small. In contrast, we would not have been able to see this without subtracting out the list mean.

In [ ]:
across_list_WORD_rsa_df['same_category'] = across_list_WORD_rsa_df['category_WORD'] == across_list_WORD_rsa_df['category_WORD2']
across_list_WORD_rsa_list_df = across_list_WORD_rsa_df.groupby(
    ['subject_WORD', 'session_WORD', 'list_WORD', 'list_WORD2', 'same_category'], as_index=False
).agg({'corr_z': 'mean'})
across_list_WORD_rsa_session_df = across_list_WORD_rsa_list_df.groupby(
    ['subject_WORD', 'session_WORD', 'same_category'], as_index=False
).agg({'corr_z': 'mean'})
across_list_WORD_rsa_subject_df = across_list_WORD_rsa_session_df.groupby(
    ['subject_WORD', 'same_category'], as_index=False
).agg({'corr_z': 'mean'})

In [ ]:
g = sns.catplot(x='same_category', y='corr_z', 
            data=across_list_WORD_rsa_subject_df, kind='point')
g.set(xlabel='Same Category', ylabel='$z$-transformed Correlation')

# Serial position similarity

The next set of results we'll investigate are from [Manning et al. (2011)](https://www.pnas.org/doi/10.1073/pnas.1015174108). The first result that Manning showed was that neural activity seemed to drift such that the similarity of patterns of intracranial EEG decreased as a function of distance in the list.

<center>
    <img src="figures/MannEtal11A.png" width=400>
</center>

We'll again do this analysis within list. Try to follow what we did above but this time, instead of comparing two categories, we will compare two serial positions. The relevant columns are `serialpos_WORD` and `serialpos_WORD2`. Try to make a heatmap of all of the comparisons

From this heatmap, we can clearly see how the similarity increases as a function of distance. We can replot this data as a function of distance to look a bit more like the Manning et al. plot. The following line will help you out:

In [ ]:
within_list_WORD_rsa_df['serialpos_dist'] = within_list_WORD_rsa_df['serialpos_WORD2'] - within_list_WORD_rsa_df['serialpos_WORD']

We'll also show that being from the same category influences similarity on top of the serial position distance within a list. It's crucial to account for both at the same time because of the category structure of the list. Use the split-apply-combine technique with both `serialpos_dist` and `same_category` to get the mean within each possible combination. Use the [catplot](https://seaborn.pydata.org/generated/seaborn.catplot.html) parameter `hue` to display both variables on the same plot.